In [1]:
from pathlib import Path
import re
import pandas as pd

# Folder containing the importance-magnitude CSV files
input_folder = Path(".")

# Match files such as:
filename_pattern = re.compile(
    r"^importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_(\d+)\.csv$"
)

# Detect matching files
matching_files = []

for file_path in input_folder.glob(
    "importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_*.csv"
):
    match = filename_pattern.match(file_path.name)

    if match:
        model_number = int(match.group(1))
        matching_files.append((model_number, file_path))

# Sort by model number
matching_files.sort(key=lambda x: x[0])

print(f"Detected {len(matching_files)} importance-magnitude files:")

for model_number, file_path in matching_files:
    print(f"Model {model_number}: {file_path.name}")

if len(matching_files) == 0:
    raise FileNotFoundError(
        "No matching importance-magnitude CSV files were found."
    )

if len(matching_files) != 10:
    print(
        f"\nWarning: Expected 10 files, but detected "
        f"{len(matching_files)} files."
    )


# Read and combine all files
all_importance_tables = []

for model_number, file_path in matching_files:

    df = pd.read_csv(file_path)

    # Check that the expected columns exist
    required_columns = {"Feature", "Importance Magnitude"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{file_path.name} does not contain the required columns: "
            f"'Feature' and 'Importance Magnitude'."
        )

    # Keep only the required columns
    df = df[["Feature", "Importance Magnitude"]].copy()

    # Convert importance values to numeric
    df["Importance Magnitude"] = pd.to_numeric(
        df["Importance Magnitude"],
        errors="coerce"
    )

    # Add model number for tracking
    df["Model Number"] = model_number

    all_importance_tables.append(df)

# Combine all model tables
combined_df = pd.concat(
    all_importance_tables,
    ignore_index=True
)

# Calculate average importance magnitude for each feature
average_importance = (
    combined_df
    .groupby("Feature", as_index=False)
    .agg(
        Average_Importance_Magnitude=(
            "Importance Magnitude",
            "mean"
        ),
        Number_of_Models=(
            "Importance Magnitude",
            "count"
        )
    )
)

# Sort from highest to lowest average importance
average_importance = average_importance.sort_values(
    by="Average_Importance_Magnitude",
    ascending=False,
    ignore_index=True
)

# Save the final table
output_file = (
    input_folder /
    "average_importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah.csv"
)

average_importance.to_csv(output_file, index=False)

print(f"\nSaved: {output_file.name}")
print("\nAverage importance magnitude table:")
print(average_importance)

Detected 10 importance-magnitude files:
Model 38: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_38.csv
Model 46: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_46.csv
Model 47: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_47.csv
Model 52: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_52.csv
Model 54: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_54.csv
Model 57: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_57.csv
Model 59: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_59.csv
Model 75: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_75.csv
Model 77: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_77.csv
Model 95: importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah_95.csv

Saved: average_importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah.csv

Average importance magnitude table:
                 Feature  Average_Importance_Magnitude  Number_of_Models


In [3]:
import pandas as pd

# Read the average importance-magnitude CSV file
df = pd.read_csv("average_importance_magnitude_cnn2_Specificity_model_ssDNA_target_ah.csv")

# Extract position and Struct from feature names
extracted = df["Feature"].str.extract(
    r"^Pos(\d+)_(Unpaired|Paired-Opening|Paired-Closing)$"
)

df["Position"] = pd.to_numeric(extracted[0], errors="coerce")
df["Struct"] = extracted[1]

# Ensure importance magnitudes are numeric
df["Average_Importance_Magnitude"] = pd.to_numeric(
    df["Average_Importance_Magnitude"],
    errors="coerce"
)

# Remove rows that don't match the expected format
df = df.dropna(
    subset=["Position", "Struct", "Average_Importance_Magnitude"]
).copy()

df["Position"] = df["Position"].astype(int)

# Define windows
windows = {
    "First_6": (1, 6),
    "Middle_8": (7, 14),
    "Last_6": (15, 20)
}

Structs = ["Unpaired", "Paired-Opening", "Paired-Closing"]

results = []

for Struct in Structs:
    Struct_df = df[df["Struct"] == Struct]

    row = {"Struct": Struct}

    for window_name, (start, end) in windows.items():
        avg = Struct_df.loc[
            Struct_df["Position"].between(start, end),
            "Average_Importance_Magnitude"
        ].mean()

        row[window_name] = avg

    results.append(row)

# Convert to DataFrame
result_df = pd.DataFrame(results)

# Save as CSV
result_df.to_csv(
    "average_importance_magnitude_by_Struct_position_range_cnn2_Specificity_model_ssDNA_target_ah.csv",
    index=False
)

print(result_df)

           Struct       First_6  Middle_8    Last_6
0        Unpaired  2.240609e-04  0.000399  0.000206
1  Paired-Opening  1.782684e-04  0.000107  0.000000
2  Paired-Closing  1.266989e-07  0.000127  0.000161
